In [4]:
import re
from sklearn.model_selection import train_test_split


In [7]:
# Load corpus.txt
with open('D:\Hackman\data\corpus.txt', 'r', encoding='utf-8') as f:
    corpus = [line.strip() for line in f if line.strip()]

# Check few samples and total length
print("Sample words:", corpus[:10])
print("Total words:", len(corpus))


Sample words: ['suburbanize', 'asmack', 'hypotypic', 'promoderationist', 'consonantly', 'philatelically', 'cacomelia', 'thicklips', 'luciferase', 'cinematography']
Total words: 49397


<>:2: SyntaxWarning: invalid escape sequence '\H'
<>:2: SyntaxWarning: invalid escape sequence '\H'
C:\Users\shriv\AppData\Local\Temp\ipykernel_23800\2356469371.py:2: SyntaxWarning: invalid escape sequence '\H'
  with open('D:\Hackman\data\corpus.txt', 'r', encoding='utf-8') as f:


In [12]:
from collections import defaultdict

grouped_train = defaultdict(list)
for w in corpus:
    grouped_train[len(w)].append(w)

for length, group in list(grouped_train.items())[:24]:
    print(f"Length {length}: {len(group)} words")
#prints count of first 5 word-length groups

Length 11: 5437 words
Length 6: 3716 words
Length 9: 6778 words
Length 16: 698 words
Length 14: 2019 words
Length 10: 6454 words
Length 8: 6306 words
Length 12: 4292 words
Length 13: 3094 words
Length 5: 2154 words
Length 18: 174 words
Length 4: 1077 words
Length 3: 310 words
Length 7: 5038 words
Length 15: 1226 words
Length 17: 375 words
Length 22: 8 words
Length 19: 88 words
Length 2: 70 words
Length 1: 23 words
Length 20: 40 words
Length 21: 16 words
Length 23: 3 words
Length 24: 1 words


In [20]:
from hmmlearn.hmm import CategoricalHMM
import numpy as np

# Build letter vocabulary with start/end tokens
letters = list("abcdefghijklmnopqrstuvwxyz")
tokens = ['<S>', '<E>']
vocab = tokens + letters
char_to_int = {c: i for i, c in enumerate(vocab)}
int_to_char = {i: c for c, i in char_to_int.items()}

In [21]:
# Encode each word as integer sequence with start/end
sequences = []
for word in corpus:
    seq = [char_to_int['<S>']] + [char_to_int[ch] for ch in word if ch in letters] + [char_to_int['<E>']]
    sequences.append(seq)

In [22]:
# Combine into numpy format
lengths = [len(seq) for seq in sequences]
X = np.concatenate([np.array(seq) for seq in sequences]).reshape(-1, 1)

print("Total sequences:", len(sequences))
print("Total observations:", len(X))
print("Sample encoded sequence:", sequences[0][:10])

Total sequences: 49397
Total observations: 570458
Sample encoded sequence: [0, 20, 22, 3, 22, 19, 3, 2, 15, 10]


In [23]:
# Train Categorical HMM
n_states = 30  # adjustable (20–40 recommended)
model = CategoricalHMM(n_components=n_states, n_iter=30, random_state=42, verbose=True)
model.fit(X, lengths)

print("HMM training complete!")
print("Number of states:", model.n_components)

         1 -1939612.25255166             +nan
         2 -1553842.08761486 +385770.16493680
         3 -1512048.36631652  +41793.72129833
         4 -1485733.11197270  +26315.25434382
         5 -1467531.71113471  +18201.40083800
         6 -1450681.53148376  +16850.17965094
         7 -1435867.63730930  +14813.89417447
         8 -1423179.83885038  +12687.79845891
         9 -1412632.46795902  +10547.37089137
        10 -1403706.96731498   +8925.50064404
        11 -1395928.67219143   +7778.29512355
        12 -1389155.97502264   +6772.69716879
        13 -1383112.39149246   +6043.58353018
        14 -1377714.30250863   +5398.08898383
        15 -1372845.30682916   +4868.99567947
        16 -1368331.25206764   +4514.05476152
        17 -1364100.89417592   +4230.35789172
        18 -1360233.77860452   +3867.11557140
        19 -1356811.17011989   +3422.60848463
        20 -1353720.41153443   +3090.75858545
        21 -1350726.45001655   +2993.96151788
        22 -1347743.03501856   +29

HMM training complete!
Number of states: 30


        30 -1328309.16061439   +2351.12218030


In [24]:
print("Start probabilities shape:", model.startprob_.shape)
print("Transition matrix shape:", model.transmat_.shape)
print("Emission matrix shape:", model.emissionprob_.shape)

print("\nSample start probabilities:", model.startprob_[:5])
print("\nSample transition probabilities:\n", model.transmat_[:5, :5])
print("\nSample emission probabilities:\n", model.emissionprob_[:5, :5])


Start probabilities shape: (30,)
Transition matrix shape: (30, 30)
Emission matrix shape: (30, 28)

Sample start probabilities: [0. 0. 0. 0. 0.]

Sample transition probabilities:
 [[9.22421803e-15 7.17311598e-02 4.04311815e-13 5.28096539e-28
  1.42175622e-30]
 [4.89186446e-05 3.98652067e-15 2.42043589e-20 1.70273501e-04
  1.42638538e-67]
 [1.07715719e-03 2.03945091e-16 1.29370797e-03 4.20617297e-12
  3.70480645e-31]
 [1.00237391e-02 1.00316592e-51 3.05804841e-12 2.30723502e-04
  1.84559181e-11]
 [9.76897448e-17 3.27065291e-01 1.97919576e-15 3.09171804e-07
  7.89652117e-16]]

Sample emission probabilities:
 [[0.00000000e+00 1.49899257e-07 6.12840482e-03 3.80363658e-02
  1.26313159e-01]
 [0.00000000e+00 2.13085558e-04 2.18505446e-01 1.05387702e-04
  2.76739410e-12]
 [0.00000000e+00 7.25634593e-10 5.06554581e-07 2.56462005e-02
  1.03063135e-01]
 [0.00000000e+00 1.39288241e-06 1.96865648e-05 4.29218619e-06
  4.04740440e-08]
 [0.00000000e+00 8.62668339e-02 4.61874939e-02 4.88107233e-06
  7.

In [25]:
test_word = "machine"
seq = [char_to_int['<S>']] + [char_to_int[ch] for ch in test_word if ch in letters] + [char_to_int['<E>']]
X_test = np.array(seq).reshape(-1, 1)

log_likelihood = model.score(X_test)
print(f"Log-likelihood of the word '{test_word}':", log_likelihood)


Log-likelihood of the word 'machine': -15.477691104897717


In [27]:
import joblib

# Save the model
joblib.dump(model, "../models/hmm_model.pkl")

print("HMM model saved successfully!")


FileNotFoundError: [Errno 2] No such file or directory: '../models/hmm_model.pkl'